# Phase 1 — Data Preparation & Preprocessing
Clean the 100k-row `supply_chain_disruption_recovery.csv` and save a validated, analysis-ready table to `data/processed/disruptions_clean.parquet` for use by the Phase 2 ML notebook.

Cleaning logic is adapted from `sql/normalise_load.py` (schema check, whitespace stripping, boolean/numeric normalization, dedup), decoupled from the MySQL/RDS pipeline — this notebook only reads the local CSV.

In [1]:
import pandas as pd
from pathlib import Path

CSV_PATH = Path("supply_chain_disruption_recovery.csv")
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "disruptions_clean.parquet"

## Load raw CSV

In [2]:
df = pd.read_csv(CSV_PATH)
print(f"Original rows: {len(df):,}")
print(f"Original columns: {len(df.columns)}")
df.head()

Original rows: 100,000
Original columns: 15


,disruption_id,disruption_type,industry,supplier_tier,supplier_region,supplier_size,has_backup_supplier,disruption_severity,production_impact_pct,revenue_loss_usd,response_type,response_time_days,partial_recovery_days,full_recovery_days,permanent_supplier_change
0,DIS-000001,Factory Incident,Pharmaceuticals,1,Asia-Pacific,Large,False,2,54.3,11376013.16,Customer Delay,14,3,59,False
1,DIS-000002,Cyber Attack,Electronics,2,Asia-Pacific,Medium,True,4,52.4,1199054.59,Alternative Supplier,1,29,97,True
2,DIS-000003,Port Congestion,Electronics,1,Asia-Pacific,Small,True,2,32.4,228848.77,Alternative Supplier,1,18,67,True
3,DIS-000004,Labor Strike,Electronics,2,Europe,Large,True,1,15.7,2257287.82,Alternative Supplier,1,18,36,False
4,DIS-000005,Geopolitical,Electronics,1,South America,Medium,True,2,32.9,965588.44,Alternative Supplier,7,11,61,True


## Schema check & column normalization

In [3]:
REQUIRED_COLUMNS = [
    "disruption_id",
    "disruption_type",
    "industry",
    "supplier_tier",
    "supplier_region",
    "supplier_size",
    "has_backup_supplier",
    "disruption_severity",
    "production_impact_pct",
    "revenue_loss_usd",
    "response_type",
    "response_time_days",
    "partial_recovery_days",
    "full_recovery_days",
    "permanent_supplier_change",
]

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

missing = set(REQUIRED_COLUMNS) - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

df = df[REQUIRED_COLUMNS].copy()
print("All required columns present.")

All required columns present.


## Normalize string, boolean, and numeric fields

In [4]:
def normalize_boolean(value):
    """Convert common CSV representations into Python booleans."""
    if pd.isna(value):
        return None
    value = str(value).strip().lower()
    if value in {"true", "1", "yes", "y", "t"}:
        return True
    if value in {"false", "0", "no", "n", "f"}:
        return False
    raise ValueError(f"Invalid boolean value: {value}")


string_columns = [
    "disruption_id",
    "disruption_type",
    "industry",
    "supplier_region",
    "supplier_size",
    "response_type",
]
for column in string_columns:
    df[column] = df[column].astype("string").str.strip()

# supplier_tier (1-4) and disruption_severity (1-5) are small integer codes
df["supplier_tier"] = pd.to_numeric(df["supplier_tier"], errors="coerce").astype("Int64")
df["disruption_severity"] = pd.to_numeric(df["disruption_severity"], errors="coerce").astype("Int64")

df["has_backup_supplier"] = df["has_backup_supplier"].apply(normalize_boolean)
df["permanent_supplier_change"] = df["permanent_supplier_change"].apply(normalize_boolean)

numeric_columns = [
    "production_impact_pct",
    "revenue_loss_usd",
    "response_time_days",
    "partial_recovery_days",
    "full_recovery_days",
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print("Dtypes after normalization:")
df.dtypes

Dtypes after normalization:


disruption_id                 string
disruption_type               string
industry                      string
supplier_tier                  Int64
supplier_region               string
supplier_size                 string
has_backup_supplier             bool
disruption_severity            Int64
production_impact_pct        float64
revenue_loss_usd             float64
response_type                 string
response_time_days             int64
partial_recovery_days          int64
full_recovery_days             int64
permanent_supplier_change       bool
dtype: object

## Drop missing IDs & duplicate records

In [5]:
before = len(df)
df = df.dropna(subset=["disruption_id"])
df = df.drop_duplicates(subset=["disruption_id"], keep="first")
after = len(df)
print(f"Rows before: {before:,}")
print(f"Rows after dropping missing/duplicate IDs: {after:,}")
print(f"Rows removed: {before - after:,}")

Rows before: 100,000
Rows after dropping missing/duplicate IDs: 100,000
Rows removed: 0


## Validation summary

In [6]:
print("Validation")
print(f"Rows: {len(df):,}")

duplicate_count = df["disruption_id"].duplicated().sum()
print(f"Duplicate IDs remaining: {duplicate_count:,}")

print("\nNull values per column:")
nulls = df.isnull().sum()
print(nulls[nulls > 0].sort_values(ascending=False) if nulls.sum() else "None")

invalid_impact = df[(df["production_impact_pct"] < 0) | (df["production_impact_pct"] > 100)]
print(f"\nInvalid production_impact_pct rows (outside 0-100): {len(invalid_impact):,}")

invalid_loss = df[df["revenue_loss_usd"] < 0]
print(f"Negative revenue_loss_usd rows: {len(invalid_loss):,}")

invalid_tier = df[~df["supplier_tier"].isin([1, 2, 3, 4])]
print(f"supplier_tier outside 1-4: {len(invalid_tier):,}")

invalid_severity = df[~df["disruption_severity"].isin([1, 2, 3, 4, 5])]
print(f"disruption_severity outside 1-5: {len(invalid_severity):,}")

Validation
Rows: 100,000
Duplicate IDs remaining: 0

Null values per column:
None

Invalid production_impact_pct rows (outside 0-100): 0
Negative revenue_loss_usd rows: 0
supplier_tier outside 1-4: 0
disruption_severity outside 1-5: 0


## Save cleaned dataset

In [7]:
df.to_parquet(OUT_PATH, index=False)
print(f"Saved cleaned dataset to {OUT_PATH} ({len(df):,} rows, {len(df.columns)} columns)")
df.head()

Saved cleaned dataset to data\processed\disruptions_clean.parquet (100,000 rows, 15 columns)


,disruption_id,disruption_type,industry,supplier_tier,supplier_region,supplier_size,has_backup_supplier,disruption_severity,production_impact_pct,revenue_loss_usd,response_type,response_time_days,partial_recovery_days,full_recovery_days,permanent_supplier_change
0,DIS-000001,Factory Incident,Pharmaceuticals,1,Asia-Pacific,Large,False,2,54.3,11376013.16,Customer Delay,14,3,59,False
1,DIS-000002,Cyber Attack,Electronics,2,Asia-Pacific,Medium,True,4,52.4,1199054.59,Alternative Supplier,1,29,97,True
2,DIS-000003,Port Congestion,Electronics,1,Asia-Pacific,Small,True,2,32.4,228848.77,Alternative Supplier,1,18,67,True
3,DIS-000004,Labor Strike,Electronics,2,Europe,Large,True,1,15.7,2257287.82,Alternative Supplier,1,18,36,False
4,DIS-000005,Geopolitical,Electronics,1,South America,Medium,True,2,32.9,965588.44,Alternative Supplier,7,11,61,True
